# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mansi-cs/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
!git clone https://github.com/mansi-cs/flyrank-ml-internship.git
%cd flyrank-ml-internship
!python scripts/01_prepare_features.py
!python scripts/ml_utils.py


Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 172 (delta 77), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1.86 MiB | 8.56 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/flyrank-ml-internship/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [27]:

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from scripts.ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES)

In [14]:
df = pd.read_csv("data/processed/refresh_feature_vector.csv")

print(df.shape)
print(df["is_declining_label"].value_counts())

(30000, 52)
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My Week 4 baseline used a transparent rule based on staleness, visibility, and CTR. For Week 5 I selected Logistic Regression because it produces probabilities that can be used for ranking refresh opportunities and is easy to interpret. I chose a simple model first so that any improvement over the baseline can be clearly explained rather than relying on model complexity.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
from sklearn.model_selection import GroupShuffleSplit

groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(df, groups=groups)
)

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

A grouped split by client_id was used so that pages from the same client do not appear in both train and test sets. This provides a more honest estimate of performance on unseen clients.

In [21]:
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
low_ctr = (df["ctr"] < df["ctr"].median()).astype(int)

df["baseline_action_score"] = (
    stale * 2 +
    visible * 2 +
    low_ctr
)
print("baseline_action_score" in df.columns)
train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]


True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [18]:
features = (
    MODEL_NUMERIC_FEATURES +
    MODEL_CATEGORICAL_FEATURES
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median"))
            ]),
            MODEL_NUMERIC_FEATURES
        ),
        (
            "cat",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]),
            MODEL_CATEGORICAL_FEATURES
        )
    ]
)

model = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=1000))
])

In [9]:
# Precision@K
import numpy as np

def precision_at_k(y_true, scores, k):
    idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[idx].mean()

In [28]:
#model score
model_scores = model.predict_proba(X_test)[:, 1]

p20_model = precision_at_k(y_test, model_scores, 20)
p50_model = precision_at_k(y_test, model_scores, 50)

NotFittedError: Pipeline is not fitted yet.

In [23]:
baseline_scores = test_df["baseline_action_score"]
p20_baseline = precision_at_k(
    y_test,
    baseline_scores,
    20
)

p50_baseline = precision_at_k(
    y_test,
    baseline_scores,
    50
)

print("Baseline Precision@20:", round(p20_baseline,3))
print("Baseline Precision@50:", round(p50_baseline,3))

Baseline Precision@20: 0.7
Baseline Precision@50: 0.66


In [24]:
import pandas as pd

comparison = pd.DataFrame({
    "Method": [
        "Week 4 Baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        p20_baseline,
        p20_model
    ],
    "Precision@50": [
        p50_baseline,
        p50_model
    ]
})

comparison

,Method,Precision@20,Precision@50
0,Week 4 Baseline,0.7,0.66
1,Logistic Regression,0.9,0.80


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [26]:
predictions = model.predict(X_test)

results = test_df.copy()

results["actual"] = y_test.values
results["predicted"] = predictions
results["probability"] = model_scores

NotFittedError: Pipeline is not fitted yet.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.